# 01 — Data Pipeline: RepoBench Embeddings & Triplets

This notebook prepares training data for the HCCS scorer from RepoBench v1.1.

**What it produces:**
- `query_embeddings.pt` — CodeBERT embeddings for all 8,033 queries
- `chunk_embeddings.pt` — CodeBERT embeddings for all context chunks
- `gold_indices.pt` — gold snippet indices from the dataset
- `data/triplets.jsonl` — ~70K–80K contrastive triplets

**How it works:**
1. Load RepoBench v1.1 (`tianyang/repobench_python_v1.1`, split `cross_file_first`)
2. Pre-compute CodeBERT embeddings for all queries and chunks (~2hrs on T4)
3. Generate triplets using `gold_snippet_index` (instant, CPU only)
4. Save everything to Drive

**Runtime:** ~2 hours on T4 (mostly embedding computation)

---
All cells are **idempotent** — safe to re-run after disconnect.

## Setup

In [ ]:
!pip install torch transformers datasets rank-bm25 tqdm --quiet

In [ ]:
import sys, os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_DIR = '/content/drive/MyDrive/HaluGuard'
except ImportError:
    REPO_DIR = os.path.abspath('..')

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
!pip install -e '.[dev]' -q

from notebooks.utils import check_gpu, print_env_summary, get_drive_path
DEVICE = check_gpu()
print_env_summary()

In [ ]:
from pathlib import Path

import numpy as np
import torch
from tqdm import tqdm

from haluguard.data_pipeline import (
    create_all_triplets,
    load_triplets,
    save_triplets,
    summarise_triplets,
)

DRIVE_DIR = get_drive_path('HaluGuard/data')
EMB_DIR   = DRIVE_DIR / 'embeddings'
EMB_DIR.mkdir(parents=True, exist_ok=True)
TRIPLETS_PATH = DRIVE_DIR / 'triplets.jsonl'
print(f'Data dir: {DRIVE_DIR}')
print(f'Embeddings dir: {EMB_DIR}')

## 1. Load and Inspect RepoBench v1.1

In [ ]:
from datasets import load_dataset

ds = load_dataset('tianyang/repobench_python_v1.1', split='cross_file_first')
print(f'Dataset size: {len(ds)} examples')
print(f'Columns: {ds.column_names}')

In [ ]:
# Inspect first 3 examples
for i in range(3):
    ex = ds[i]
    print(f'\n--- Example {i} ---')
    print(f'repo_name: {ex["repo_name"]}')
    print(f'file_path: {ex["file_path"]}')
    print(f'level: {ex["level"]}')
    print(f'Chunks: {len(ex["context"])}, Gold index: {ex["gold_snippet_index"]}')
    print(f'Next line: {ex["next_line"]!r}')
    print(f'Gold snippet path: {ex["context"][ex["gold_snippet_index"]]["path"]}')
    print(f'Gold snippet preview: {ex["context"][ex["gold_snippet_index"]]["snippet"][:120]}...')
    print(f'Cropped code (last 80 chars): ...{ex["cropped_code"][-80:]}')

## 2. Pre-compute CodeBERT Embeddings

Embed all queries (`cropped_code`) and all context snippets using frozen CodeBERT.
This takes ~2 hours on T4 and is the most expensive step. Results are cached.

In [ ]:
from transformers import AutoModel, AutoTokenizer

ENCODER_NAME = 'microsoft/codebert-base'
tokenizer = AutoTokenizer.from_pretrained(ENCODER_NAME)
codebert = AutoModel.from_pretrained(ENCODER_NAME).to(DEVICE)
codebert.eval()
print(f'CodeBERT loaded on {DEVICE}')

In [ ]:
from haluguard.hccs import embed_code

QUERY_EMB_PATH = EMB_DIR / 'query_embeddings.pt'

if QUERY_EMB_PATH.exists():
    query_embs = torch.load(QUERY_EMB_PATH)
    print(f'Loaded cached query embeddings: {query_embs.shape}')
else:
    print('Computing query embeddings...')
    query_embs_list = []
    for i, ex in enumerate(tqdm(ds, desc='Embedding queries')):
        emb = embed_code(ex['cropped_code'], tokenizer, codebert, device=DEVICE)
        query_embs_list.append(torch.from_numpy(emb))
        if (i + 1) % 500 == 0:
            print(f'  Progress: {i + 1}/{len(ds)}')
    query_embs = torch.stack(query_embs_list)  # [8033, 768]
    torch.save(query_embs, QUERY_EMB_PATH)
    print(f'Saved query embeddings: {query_embs.shape}')

In [ ]:
CHUNK_EMB_PATH = EMB_DIR / 'chunk_embeddings.pt'

if CHUNK_EMB_PATH.exists():
    chunk_embs = torch.load(CHUNK_EMB_PATH)
    print(f'Loaded cached chunk embeddings: {len(chunk_embs)} examples')
else:
    print('Computing chunk embeddings (this takes ~1-2 hours)...')
    chunk_embs = []  # list of tensors, one per example
    for i, ex in enumerate(tqdm(ds, desc='Embedding chunks')):
        snippets = [c['snippet'] for c in ex['context']]
        if snippets:
            from haluguard.hccs import batch_embed
            embs = batch_embed(snippets, tokenizer, codebert, device=DEVICE)
            chunk_embs.append(torch.from_numpy(embs))
        else:
            chunk_embs.append(torch.empty(0, 768))
        if (i + 1) % 500 == 0:
            print(f'  Progress: {i + 1}/{len(ds)}')
    torch.save(chunk_embs, CHUNK_EMB_PATH)
    print(f'Saved chunk embeddings for {len(chunk_embs)} examples')

In [ ]:
GOLD_IDX_PATH = EMB_DIR / 'gold_indices.pt'

gold_indices = [ex['gold_snippet_index'] for ex in ds]
torch.save(gold_indices, GOLD_IDX_PATH)
print(f'Saved {len(gold_indices)} gold indices')
print(f'Valid gold indices: {sum(1 for i, ex in enumerate(ds) if 0 <= gold_indices[i] < len(ex["context"]))}/{len(ds)}')

## 3. Generate Contrastive Triplets

Using `gold_snippet_index` directly — no execution needed.
For each example: positive = `context[gold_snippet_index]`, negatives = all others.

In [ ]:
triplets = create_all_triplets(ds, max_negatives=None, seed=42)
print(f'Generated {len(triplets)} triplets')

# Save to disk
save_triplets(triplets, TRIPLETS_PATH)
print(f'Saved to {TRIPLETS_PATH}')

## 4. Analyse Triplets

In [ ]:
if TRIPLETS_PATH.exists():
    triplets = load_triplets(TRIPLETS_PATH)
    summary  = summarise_triplets(triplets)
    print(f'Total triplets:          {summary["total"]}')
    print(f'Unique tasks:            {summary["unique_tasks"]}')
    print(f'Avg negatives per task:  {summary["avg_negatives_per_task"]}')

    if summary['total'] >= 1000:
        print('\nReady for training! Run notebook 02 next.')
    else:
        print('\nWARNING: fewer than 1000 triplets — check dataset loading.')
else:
    print('No triplets yet — run the generation cell above first.')

In [ ]:
# Verify all files are saved
for path in [QUERY_EMB_PATH, CHUNK_EMB_PATH, GOLD_IDX_PATH, TRIPLETS_PATH]:
    exists = path.exists()
    size_mb = path.stat().st_size / 1e6 if exists else 0
    print(f'  {"OK" if exists else "MISSING"}: {path.name} ({size_mb:.1f} MB)')